## The Robotic Cell Problem (fixed job order)

From Minoli & Righini (2026), European Journal of Operational Research.

**Setup**
- There are **n jobs** and **m machines** in a line.
- Every job starts at the **source node** (station 0) and has to go through machine 1, then 2, ..., then m, and end at the **sink node** (station m+1).
- One robot moves all the jobs between stations. It can only carry one job at a time.


**Data**

| Symbol | Meaning |
|---|---|
| $m$ | number of machines |
| $n$ | number of jobs |
| $p_{i,k} \in \mathbb{N}_+$ | processing time of job $k$ on machine $i$ (loading/unloading time included) |
| $\tau_{i,j} \in \mathbb{N}$ | robot travel time between stations $i$ and $j$; satisfies the triangle inequality, and is the same whether the robot is loaded or empty |

**Rules**
- The job order is given: job 1 goes first, then job 2, and so on.
- The machine order is given: every job goes 1, 2, ..., m.
- Each machine can only hold one job at a time and there is no space between machines.
- A job can't leave a machine until it's done processing.
- A finished job stays on its machine until the robot picks it up.
- The robot can only move a job to the next machine if that machine is empty.
- The source and sink can hold any number of jobs.

**Decision**

Since the job order and machine order are given, the only decision is the order of the robot's moves, i.e. which job it picks up and moves next.

**Objective**

Minimize the makespan, which is the time when the last job reaches the sink.

**Why it's hard**

The robot has to balance moving jobs forward to free up machines against driving back and forth. If it waits for a slow machine it wastes time, but if it leaves a machine empty that machine is idle. The number of possible move sequences grows very fast, and the paper found that Gurobi already struggles with 8 jobs and 7 machines.

**Example (from the paper)**

m = 3 machines, n = 4 jobs

Processing times:

| machine \ job | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| 1 | 42 | 20 | 51 | 84 |
| 2 | 7 | 10 | 69 | 13 |
| 3 | 47 | 75 | 8 | 65 |

Travel times (0 = source, 4 = sink):

| | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| 0 | 0 | 11 | 25 | 23 | 10 |
| 1 | 11 | 0 | 16 | 18 | 9 |
| 2 | 25 | 16 | 0 | 19 | 20 |
| 3 | 23 | 18 | 19 | 0 | 14 |
| 4 | 10 | 9 | 20 | 14 | 0 |

Results from the paper:
- Moving one job all the way through at a time: 761
- Moving all the jobs forward one machine at a time: 607
- Optimal: 553

So the order of the robot's moves makes a big difference even though the jobs and machines are the same.

## DIDP Model: Grace's Method

**Parameters**
- $m$ = number of machines, $n$ = number of jobs
- Stations: $\mathcal{M} = \{0, 1, \dots, m+1\}$, where $0$ is the source node, $1$ to $m$ are the machines, and $m+1$ is the sink node
- $p_{i,k}$ = processing time of job $k$ on machine $i$
- $\tau_{i,j}$ = time for the robot to travel from station $i$ to station $j$, for $i, j \in \mathcal{M}$

**State**
- $robot$ = station the robot is at (starts at $0$)
- $next$ = next job waiting at the source (starts at $1$, $n+1$ means no jobs left)
- $on_i$ = job on machine $i$ ($0$ means empty)
- $left_i$ = processing time left on machine $i$

**Transitions**

For each station $s = 0, \dots, m$, the robot goes to station $s$, waits for the job to finish, and moves it to $s+1$.

Time of the move:
- If $s = 0$: $\quad time_s = \tau_{robot,0} + \tau_{0,1}$
- Otherwise: $\quad time_s = \max(\tau_{robot,s},\ left_s) + \tau_{s,s+1}$

Case 1 ($s = 0$): take the next job from the source
- Preconditions: $next \le n$ and $on_1 = 0$
- Effects: $on_1 = next$, $\ left_1 = p_{1,next}$, $\ next = next + 1$

Case 2 ($s = m$): move the job on the last machine to the sink
- Preconditions: $on_m \ne 0$
- Effects: $on_m = 0$, $\ left_m = 0$

Case 3 ($1 \le s < m$): move the job on machine $s$ to machine $s+1$
- Preconditions: $on_s \ne 0$ and $on_{s+1} = 0$ (next machine has to be empty)
- Effects: $on_{s+1} = on_s$, $\ left_{s+1} = p_{s+1,\,on_s}$, $\ on_s = 0$, $\ left_s = 0$

For all cases:
- $robot = s+1$
- For every other machine $i$: $\ left_i = \max(left_i - time_s,\ 0)$ since it keeps processing while the robot moves

**Base case**

$next = n+1$ and $on_i = 0$ for all machines $i$, then the cost is $0$

**Bellman equation**

$$V(robot,\ next,\ on,\ left) = \min_{s} \big\{\, time_s + V(\text{state after move } s) \,\big\}$$

where $s$ is any station that meets the preconditions, and $V = 0$ at the base case.

Optimal makespan $= V(0,\ 1,\ \mathbf{0},\ \mathbf{0})$

**Dual bound:** $0$ since time can't be negative

In [3]:
import didppy as dp
model = dp.Model(maximize=False)    # minimization problem (minimize makespan), costs are integers

# Parameters
m = 3     # number of machines
n = 4     # number of jobs
# Stations: 0 = source node, 1..m = machines, m+1 = sink node

# p[i, k] = processing time of job k on machine i
#   row 0 and column 0 are dummies (the source does no processing, job 0 means "no job")
p = model.add_int_table([[0,  0,  0,  0,  0],     # station 0 (source)
                         [0, 42, 20, 51, 84],     # machine 1
                         [0,  7, 10, 69, 13],     # machine 2
                         [0, 47, 75,  8, 65]])    # machine 3

# tau[i, j] = robot travel time from station i to station j (0 = source, 4 = sink)
tau = model.add_int_table([[ 0, 11, 25, 23, 10],
                           [11,  0, 16, 18,  9],
                           [25, 16,  0, 19, 20],
                           [23, 18, 19,  0, 14],
                           [10,  9, 20, 14,  0]])

# Object Types
station = model.add_object_type(number=m+2)    # stations 0 .. m+1
job = model.add_object_type(number=n+2)        # jobs 0 .. n+1  (0 = no job, n+1 = no jobs left at the source)

# State
robot = model.add_element_var(object_type=station, target=0)    # robot: station the robot is at, starts at the source
next = model.add_element_var(object_type=job, target=1)         # next: next job waiting at the source, starts with job 1
on = [None]      # on[i]: job on machine i (0 = empty)
left = [None]    # left[i]: processing time left on machine i
for i in range(1, m+1):
    on.append(model.add_element_var(object_type=job, target=0))    # all machines start empty
    left.append(model.add_int_var(target=0))                       # nothing left to process at the start

# Bellman
# One transition per station s = 0..m: "robot goes to station s, waits for the job, moves it to s+1"
for s in range(m+1):

    # time of the move = max(travel to station s, wait for the job on s) + travel from s to s+1
    if s == 0:
        time = tau[robot, 0] + tau[0, 1]                             # jobs at the source are always ready
    else:
        time = dp.max(tau[robot, s], left[s]) + tau[s, s+1]

    # CASE 1: take the next job from the source and put it on machine 1
    if s == 0:
        preconditions = [next <= n, on[1] == 0]                      # a job is left AND machine 1 is empty
        effects = [(on[1], next),                                    # job goes on machine 1
                   (left[1], p[1, next]),                            # machine 1 starts processing it
                   (next, next + 1)]                                 # the next job in line moves up

    # CASE 2: move the job on the last machine to the sink (always has room)
    elif s == m:
        preconditions = [on[m] != 0]                                 # machine m has a job
        effects = [(on[m], 0), (left[m], 0)]                         # machine m becomes empty

    # CASE 3: move the job on machine s to machine s+1
    else:
        preconditions = [on[s] != 0, on[s+1] == 0]                   # machine s has a job AND machine s+1 is empty
        effects = [(on[s+1], on[s]),                                 # job moves to machine s+1
                   (left[s+1], p[s+1, on[s]]),                       # machine s+1 starts processing it
                   (on[s], 0), (left[s], 0)]                         # machine s becomes empty

    # FOR ALL CASES
    effects.append((robot, s+1))                                     # robot ends up at station s+1
    for i in range(1, m+1):                                          # every other machine keeps processing
        if i != s and i != s+1:
            effects.append((left[i], dp.max(left[i] - time, 0)))

    move = dp.Transition(name = "move {} -> {}".format(s, s+1),
                         cost = time + dp.IntExpr.state_cost(),     # makespan = sum of the move times
                         effects = effects,
                         preconditions = preconditions,)
    model.add_transition(move)

# Base case: no jobs left at the source AND every machine is empty (all jobs are at the sink)
model.add_base_case([next == n+1] + [on[i] == 0 for i in range(1, m+1)])

model.add_dual_bound(0)   # time can't be negative so 0 is a valid lower bound


# Solve
solver = dp.CABS(model)
solution = solver.search()

# Print
print("Solution:")
step = 1
for t in solution.transitions:
    print("Step: ", step, " Robot: ", t.name)
    step += 1
print("")
print("Makespan: {}".format(solution.cost))

if solution.is_optimal:
    print("The optimality is proved")
else:
    print("Best bound: {}".format(solution.best_bound))

Solution:
Step:  1  Robot:  move 0 -> 1
Step:  2  Robot:  move 1 -> 2
Step:  3  Robot:  move 2 -> 3
Step:  4  Robot:  move 0 -> 1
Step:  5  Robot:  move 1 -> 2
Step:  6  Robot:  move 3 -> 4
Step:  7  Robot:  move 0 -> 1
Step:  8  Robot:  move 2 -> 3
Step:  9  Robot:  move 1 -> 2
Step:  10  Robot:  move 0 -> 1
Step:  11  Robot:  move 3 -> 4
Step:  12  Robot:  move 2 -> 3
Step:  13  Robot:  move 3 -> 4
Step:  14  Robot:  move 1 -> 2
Step:  15  Robot:  move 2 -> 3
Step:  16  Robot:  move 3 -> 4

Makespan: 553
The optimality is proved


## DIDP Model 2: Paper's Method

This version copies the paper's state and update rules (Section 4, Eqs. 9 to 12) as closely as DIDP allows.

**Parameters** (same as before)
- $m$ = number of machines, $n$ = number of jobs
- Stations: $\mathcal{M} = \{0, 1, \dots, m+1\}$, where $0$ is the source node, $1$ to $m$ are the machines, and $m+1$ is the sink node
- $p_{i,k}$ = processing time of job $k$ on machine $i$
- $\tau_{i,j}$ = time for the robot to travel from station $i$ to station $j$, for $i, j \in \mathcal{M}$
- $\infty$ = a big number used for "nothing is here"

**State** (the paper's state $\langle t, y, (x_0, \dots, x_m), (C_0, \dots, C_m) \rangle$)
- $t$ = current time (the clock)
- $y$ = station the robot is at
- $x_i$ = job at station $i$ ($0$ means empty). For the source, $x_0$ is the next job waiting there.
- $C_i$ = time when the job at station $i$ is finished ($\infty$ if station $i$ is empty). For the source, $C_0$ is when the next job is ready.

Start: $t = 0$, $y = 0$, $x = (1, 0, \dots, 0)$, $C = (0, \infty, \dots, \infty)$

**Main difference from Model 1**

Model 1 tracked the time *left* on each machine. The paper tracks the actual clock time $t$ and the actual finish time $C_i$ of each job. Both describe the same situation.

**Transitions**

For each station $s = 0, \dots, m$ the robot does three steps:
1. go from $y$ to station $s$
2. wait for the job $k = x_s$ there to finish, if needed
3. carry job $k$ from $s$ to $s+1$

Preconditions: station $s$ is not empty ($x_s \ne 0$) and not blocked ($s = m$ or $x_{s+1} = 0$)

New time (Eq. 9):

$$t' = \max(t + \tau_{y,s},\ C_s) + \tau_{s,s+1}$$

New robot position (Eq. 10): $y' = s+1$

Updates to $x$ and $C$ (Eqs. 11 and 12):
- **Case I**, the station the job leaves (a machine, or the source when $k = n$ is the last job): $x_s' = 0$, $C_s' = \infty$
- **Case II**, the source when $k < n$ (more jobs are waiting): $x_0' = k + 1$, $C_0' = t'$
- **Case III**, the station that gets the job (if $s < m$): $x_{s+1}' = k$, $C_{s+1}' = t' + p_{s+1,k}$
- **Case IV**, all other stations: no change

In the code, the source has two transitions, one for Case II (not the last job) and one for Case I (the last job).

**Cost of a move**

The cost is how much the clock went up, $t' - t$. So when all the jobs are done, the total cost equals the final $t$, which is the makespan.

**Base case**

$x_i = 0$ for all $i = 0, \dots, m$ (nothing at the source or on any machine, so every job is at the sink). The cost is $0$.

**Bellman equation**

$$V(t, y, x, C) = \min_{s} \big\{\, (t' - t) + V(t', y', x', C') \,\big\}$$

where $s$ is any station that meets the preconditions, and $V = 0$ at the base case.

Optimal makespan $= V(0,\ 0,\ (1, 0, \dots, 0),\ (0, \infty, \dots, \infty))$

The paper writes this forward instead: it keeps extending states from the start and the answer is the smallest $t$ among the final states. It gives the same answer.

**Dominance**

The paper removes a state if another state is at least as good. Two states with the same jobs in the same places and the robot at the same station: the one with the earlier time and earlier finish times is better. In DIDP this is done by making $t$ and $C_i$ resource variables with "less is better".

The paper's full dominance rule (Eq. 17) can also compare states where the jobs are in different places, using lower bounds on when each job can get somewhere. DIDP can't express that kind of rule, so this version only uses the simpler one.

**Dual bound:** $0$ since time can't be negative

In [4]:
import didppy as dp
model = dp.Model(maximize=False)    # minimization problem (minimize makespan), costs are integers

# Parameters
m = 3     # number of machines
n = 4     # number of jobs
# Stations: 0 = source node, 1..m = machines, m+1 = sink node
INF = 100000    # "infinity": finish time of an empty station

# p[i, k] = processing time of job k on machine i
#   row 0 and column 0 are dummies (the source does no processing, job 0 means "no job")
p = model.add_int_table([[0,  0,  0,  0,  0],     # station 0 (source)
                         [0, 42, 20, 51, 84],     # machine 1
                         [0,  7, 10, 69, 13],     # machine 2
                         [0, 47, 75,  8, 65]])    # machine 3

# tau[i, j] = robot travel time from station i to station j (0 = source, 4 = sink)
tau = model.add_int_table([[ 0, 11, 25, 23, 10],
                           [11,  0, 16, 18,  9],
                           [25, 16,  0, 19, 20],
                           [23, 18, 19,  0, 14],
                           [10,  9, 20, 14,  0]])

# Object Types
station = model.add_object_type(number=m+2)    # stations 0 .. m+1
job = model.add_object_type(number=n+1)        # jobs 0 .. n  (0 = no job)

# State: the paper's < t, y, (x_0..x_m), (C_0..C_m) >
# resource variables with less_is_better=True: if two states are the same except for these,
# the one with smaller values is at least as good (this is the dominance part)
t = model.add_int_resource_var(target=0, less_is_better=True)       # t: current time
y = model.add_element_var(object_type=station, target=0)            # y: station the robot is at
x = []   # x[i]: job at station i (0 = empty); x[0] = next job waiting at the source
C = []   # C[i]: time when the job at station i is finished (INF = empty)
for i in range(m+1):
    if i == 0:
        x.append(model.add_element_var(object_type=job, target=1))             # job 1 waits at the source
        C.append(model.add_int_resource_var(target=0, less_is_better=True))    # and is ready at time 0
    else:
        x.append(model.add_element_var(object_type=job, target=0))             # machines start empty
        C.append(model.add_int_resource_var(target=INF, less_is_better=True))

# Bellman
for s in range(m+1):

    # Eq. 9: new time = max(arrive at station s, job on s finishes) + carry it to s+1
    new_t = dp.max(t + tau[y, s], C[s]) + tau[s, s+1]

    # Preconditions: station s is not empty AND not blocked
    preconditions = [x[s] != 0]
    if s < m:
        preconditions.append(x[s+1] == 0)

    # Effects for all moves: clock and robot position (Eqs. 9 and 10)
    effects = [(t, new_t), (y, s+1)]

    # Case III: the station that gets the job (the sink needs no update)
    if s < m:
        effects += [(x[s+1], x[s]), (C[s+1], new_t + p[s+1, x[s]])]

    if s == 0:
        # Case II: take a job from the source, and it's NOT the last one
        g1 = dp.Transition(name = "move {} -> {}".format(s, s+1),
                           cost = (new_t - t) + dp.IntExpr.state_cost(),     # cost = how much the clock went up
                           effects = effects + [(x[0], x[0] + 1), (C[0], new_t)],
                           preconditions = preconditions + [x[0] < n],)
        model.add_transition(g1)

        # Case I at the source: take the LAST job, so the source is now empty
        g2 = dp.Transition(name = "move {} -> {}".format(s, s+1),
                           cost = (new_t - t) + dp.IntExpr.state_cost(),
                           effects = effects + [(x[0], 0), (C[0], INF)],
                           preconditions = preconditions + [x[0] == n],)
        model.add_transition(g2)

    else:
        # Case I: machine s becomes empty
        g3 = dp.Transition(name = "move {} -> {}".format(s, s+1),
                           cost = (new_t - t) + dp.IntExpr.state_cost(),
                           effects = effects + [(x[s], 0), (C[s], INF)],
                           preconditions = preconditions,)
        model.add_transition(g3)

    # Case IV: every other station stays the same, so nothing to write

# Base case: nothing at the source and nothing on any machine (all jobs are at the sink)
model.add_base_case([x[i] == 0 for i in range(m+1)])

model.add_dual_bound(0)   # time can't be negative so 0 is a valid lower bound


# Solve
solver = dp.CABS(model)
solution = solver.search()

# Print
print("Solution:")
step = 1
for tr in solution.transitions:
    print("Step: ", step, " Robot: ", tr.name)
    step += 1
print("")
print("Makespan: {}".format(solution.cost))

if solution.is_optimal:
    print("The optimality is proved")
else:
    print("Best bound: {}".format(solution.best_bound))

Solution:
Step:  1  Robot:  move 0 -> 1
Step:  2  Robot:  move 1 -> 2
Step:  3  Robot:  move 2 -> 3
Step:  4  Robot:  move 0 -> 1
Step:  5  Robot:  move 1 -> 2
Step:  6  Robot:  move 3 -> 4
Step:  7  Robot:  move 0 -> 1
Step:  8  Robot:  move 2 -> 3
Step:  9  Robot:  move 1 -> 2
Step:  10  Robot:  move 0 -> 1
Step:  11  Robot:  move 3 -> 4
Step:  12  Robot:  move 2 -> 3
Step:  13  Robot:  move 3 -> 4
Step:  14  Robot:  move 1 -> 2
Step:  15  Robot:  move 2 -> 3
Step:  16  Robot:  move 3 -> 4

Makespan: 553
The optimality is proved
